# 📘 Notebook Final Soigné — Pipeline Deep Learning

## Ultra Pro Access Core AI · Safe Exit Pro

| | |
|---|---|
| **Équipe** | Yassine Mokni · Hadil Dhaya |
| **Domaine** | Reconnaissance faciale · Contrôle d'accès |
| **Backbone retenu** | InceptionResnetV1 (VGGFace2) |
| **Déploiement** | `main.py` (PyQt5) + Arduino (G/D/O) |

---

Ce document présente le **pipeline de recherche et d'ingénierie** menant au système de production. Les brouillons (`notebooks/drafts/`) documentent les pistes écartées ; ce notebook consolide la solution finale.

---
# Section 1 — Prétraitement et augmentation de données

## 1.1 Structure des données

- **`dataset/<identité>/`** : images brutes capturées (≈150 par utilisateur via l'UI).
- **`database/users_embeddings.pkl`** : signatures biométriques (vecteurs 512-D normalisés).

## 1.2 Prétraitement FaceNet

Le modèle attend des entrées **160×160**, normalisées dans \([-1, 1]\) :

In [ ]:
from pathlib import Path
import os

# Racine projet (parent de notebooks/)
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / 'notebooks').exists() and not (PROJECT_ROOT / 'main.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == 'final':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATASET_DIR = PROJECT_ROOT / 'dataset'
DATABASE_DIR = PROJECT_ROOT / 'database'
DATABASE_PATH = DATABASE_DIR / 'users_embeddings.pkl'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Dataset:', DATASET_DIR)
print('Database:', DATABASE_PATH)

In [ ]:
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Pipeline identique à main.py (production)
preprocess = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

augment_train = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# Aperçu d'une image si disponible
sample = None
if DATASET_DIR.exists():
    for user in sorted(DATASET_DIR.iterdir()):
        if user.is_dir():
            for img in user.glob('*.jpg'):
                sample = img; break
        if sample: break

if sample:
    img = Image.open(sample).convert('RGB')
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(img); ax[0].set_title('Original'); ax[0].axis('off')
    t = preprocess(img).permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax[1].imshow(t.clip(0, 1)); ax[1].set_title('Après preprocess'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Aucune image dans dataset/ — enrôler via main.py')

---
# Section 2 — Construction du modèle

## 2.1 InceptionResnetV1 (FaceNet)

Nous utilisons le backbone **pré-entraîné VGGFace2** via `facenet-pytorch`. Il produit un **embedding métrique** de dimension 512, adapté à la comparaison par distance euclidienne (seuil `0.65` en production).

In [ ]:
import torch
from facenet_pytorch import InceptionResnetV1

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# Gel des poids (inference / fine-tuning léger possible)
for p in model.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Paramètres: {total:,} | Entraînables (gelés): {trainable:,}')

---
# Section 3 — Entraînement et validation

## 3.1 Stratégie retenue en production

Le déploiement (`main.py`) utilise l'**extraction d'embeddings** + **moyenne** sur 150 captures — pas un classifieur softmax lourd. Ce notebook simule une **boucle d'entraînement** sur un classifieur linéaire au-dessus des embeddings pour visualiser courbes Loss/Accuracy (validation académique).

In [ ]:
import numpy as np
import pickle
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

class EmbeddingDataset(Dataset):
    def __init__(self, root, transform, max_per_class=30):
        self.samples = []
        self.labels = []
        self.transform = transform
        classes = sorted([d.name for d in Path(root).iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for c in classes:
            paths = list((Path(root)/c).glob('*.jpg'))[:max_per_class]
            for p in paths:
                self.samples.append(p)
                self.labels.append(self.class_to_idx[c])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert('RGB')
        return self.transform(img), self.labels[idx]

if not DATASET_DIR.exists():
    print('dataset/ requis')
else:
    ds = EmbeddingDataset(DATASET_DIR, preprocess, max_per_class=25)
    if len(ds) < 4:
        print('Pas assez de données')
    else:
        n = len(ds)
        n_val = max(1, int(0.2 * n))
        train_ds, val_ds = torch.utils.data.random_split(ds, [n - n_val, n_val])
        train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=8)
        head = nn.Linear(512, len(ds.class_to_idx)).to(device)
        opt = optim.Adam(head.parameters(), lr=1e-3)
        crit = nn.CrossEntropyLoss()
        train_losses, val_accs = [], []
        for epoch in range(5):
            model.eval(); head.train(); tl = 0
            for x, y in train_loader:
                x = x.to(device)
                with torch.no_grad():
                    emb = model(x)
                logits = head(emb)
                loss = crit(logits, torch.tensor(y, device=device))
                opt.zero_grad(); loss.backward(); opt.step()
                tl += loss.item()
            train_losses.append(tl / max(1, len(train_loader)))
            head.eval(); correct = total = 0
            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(device)
                    pred = head(model(x)).argmax(1).cpu()
                    correct += (pred == torch.tensor(y)).sum().item()
                    total += len(y)
            val_accs.append(correct / max(1, total))
            print(f'Epoch {epoch+1}: loss={train_losses[-1]:.4f} val_acc={val_accs[-1]:.3f}')

In [ ]:
import matplotlib.pyplot as plt
if DATASET_DIR.exists() and len(ds) >= 4:
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(train_losses, 'o-', color='#0284c7'); ax[0].set_title('Courbe de Loss (entraînement)'); ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss')
    ax[1].plot(val_accs, 'o-', color='#059669'); ax[1].set_title('Accuracy validation'); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()

---
# Section 4 — Comparaison d'architectures

| Architecture | Pré-entraînement | Sortie | Verdict |
|--------------|------------------|--------|---------|
| CNN custom (draft 01) | Aucun | Logits | ❌ Précision insuffisante |
| ResNet18 (draft 03) | ImageNet | Logits | ⚠️ Bon mais non spécialisé visage |
| **InceptionResnetV1** | **VGGFace2** | **Embedding 512-D** | ✅ **Retenu** |

### Pourquoi InceptionResnetV1 ?

1. **Représentation métrique** : distance L2 entre embeddings ≈ similarité biométrique.
2. **Pré-entraînement visages** : VGGFace2 > ImageNet pour l'identité.
3. **Cohérence production** : même code dans `main.py` et `notebooks/build_database.py`.
4. **Performance MPS** : inférence temps réel sur Apple Silicon.

Les brouillons `01_Baseline_CNN`, `02_Data_Augmentation_LearningRates`, `03_Transfer_Learning_ResNet` documentent l'exploration.

---
# Section 5 — Interprétation des résultats

## 5.1 Matrice de confusion & courbes ROC (simulation sur embeddings)

Nous évaluons les paires **(embedding requête, centroïde utilisateur)** avec seuil L2.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc
from itertools import combinations
import numpy as np

if DATABASE_PATH.exists():
    with open(DATABASE_PATH, 'rb') as f:
        db = pickle.load(f)
    users = sorted(db.keys())
    if len(users) >= 2:
        # Matrice de confusion sur distances inter / intra classe
        y_true, y_pred = [], []
        for u in users:
            emb = db[u]['embedding']
            for v in users:
                d = np.linalg.norm(emb - db[v]['embedding'])
                y_true.append(1 if u == v else 0)
                y_pred.append(1 if d < 0.65 else 0)
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(cm, cmap='Blues')
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(['Rejeté', 'Accepté']); ax.set_yticklabels(['Négatif', 'Positif'])
        ax.set_xlabel('Prédiction'); ax.set_ylabel('Vérité')
        ax.set_title('Matrice de confusion (paires d\'identités)')
        for i in range(2):
            for j in range(2):
                ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i,j]>cm.max()/2 else 'black')
        plt.colorbar(im); plt.tight_layout(); plt.show()
        # ROC sur scores de similarité (1 - distance normalisée)
        scores, labels = [], []
        for u in users:
            for v in users:
                d = np.linalg.norm(db[u]['embedding'] - db[v]['embedding'])
                scores.append(1 - d)
                labels.append(1 if u == v else 0)
        fpr, tpr, _ = roc_curve(labels, scores)
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})', color='#0284c7')
        plt.plot([0, 1], [0, 1], '--', color='gray')
        plt.xlabel('Faux positifs'); plt.ylabel('Vrais positifs')
        plt.title('Courbe ROC — séparation des identités')
        plt.legend(); plt.tight_layout(); plt.show()
    else:
        print('Au moins 2 identités requises dans la base')
else:
    print('users_embeddings.pkl introuvable')

## 5.2 Métriques et discussion

| Métrique | Rôle dans Safe Exit Pro |
|----------|-------------------------|
| **Distance L2** | Score primaire entre embedding live et base |
| **Seuil 0.65** | Compromis sécurité / UX (ajustable) |
| **Stabilisation 5 frames** | Réduit faux déclenchements Arduino |
| **Taux FMR / FRR** | À mesurer sur jeu de test étiqueté |

### Lien avec le hardware

- **`G`** : accès accordé (LED verte)
- **`D`** : refus (LED rouge)
- **`O`** : idle / absence de visage

### Perspectives

- Courbe ROC sur captures réelles multi-postures
- Liveness detection (anti-spoof)
- Journalisation centralisée (`access_log.csv`)

---

**© Ultra Pro Access Core AI — Yassine Mokni & Hadil Dhaya**